In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 5.7 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [3]:
import os
from pyngrok import ngrok

In [4]:
ngrok.kill()

In [5]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://baritone-scarily-unmade.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://baritone-scarily-unmade.ngrok-free.dev


True

In [6]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [7]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

In [8]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"U119d07e95858d5d83008458c2d433c76","events":[{"type":"message","message":{"type":"text","id":"616923135312134727","quoteToken":"wkAFvvfgzPK6eVlnj9phNGes7CveGJ5pIXpl0f3WQ4BHBvwdDO_0tgncr3f8QTjeArd1l8XSzPiNVwGYZEO_zsS2HkjDifH7xbQ-c9zIksuy8jolW8kVyxuc57znfJIw_hpsXX8QBk4tt1u7nFp6lg","markAsReadToken":"eKKe8rS2Z5TVSXGZcxz6o4fjL67WnWwl1Or8gGgAGHLpN4gVBf3r7YaE5l3nenKmrdCqwsR90MKhT04i-BEUuIOd6klA-LNDhKCtl6CbPbki8D7JB7_-yEiN5orpwZMMoeYe0xrzgTVccg6GuZp-Ic9N4atBwwxec8f_z179zShiz1ZbduboK7m-VvAEP8Hlj_RCLAPNxLdPXvugFPEN7A","text":"AI 介紹明新科大，20字之內"},"webhookEventId":"01KT8D45GWF71PJS30BXMXPXEE","deliveryContext":{"isRedelivery":false},"timestamp":1780546147356,"source":{"type":"user","userId":"U3d0a7956380119affa33de4fb575bbcb"},"replyToken":"4c28d47d34504782b391db804d0f5e6e","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 04:09:11] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U119d07e95858d5d83008458c2d433c76","events":[{"type":"message","message":{"type":"text","id":"616923159689953574","quoteToken":"i5adReAyfPJAYIViaQJRA_MoMptpY3VyHZt9eMc27KnwN-TuozfzCDmtfw8en_rlW-ijqTN24Cr_ZbakdbDwfrhLH2P-FYiRyLIdetNMLTIgDqVxeMJ74lF2ClRmK6p0y9_lN6F3MfHfHnrL4Q6vrA","markAsReadToken":"XqmICn9dRXU2Mh9jGqlixgBlzAU9Rt9n5XxfDRosPNrWZj2SFehqbp5s7S5UOaz-WgLLYZeWTwZ3NSFztpNQbVKIMWtcEpJ7jXunmM3upthpjQF6xzN9g6IsT5xZhTjPedhqzfZLv748qwtBIe-zi-JLkhzuxNj4two5Bb3njIn_Otetw5JOqLVcqnN2Q7B0Lns0a3kObVsok8OdwcMN1Q","text":"AI 校長是誰"},"webhookEventId":"01KT8D4KTQ64RGBCESYHQT50G9","deliveryContext":{"isRedelivery":false},"timestamp":1780546162009,"source":{"type":"user","userId":"U3d0a7956380119affa33de4fb575bbcb"},"replyToken":"a73aea4ccd554a04ab709660bf98d1b5","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 04:09:25] "POST / HTTP/1.1" 200 -
